In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/HDFS-Log-Analytics/data/raw"
LOG_FILE = f"{DATA_PATH}/HDFS.log"
LABEL_FILE = f"{DATA_PATH}/anomaly_label.csv"

for file in os.listdir(DATA_PATH):
    print(file)

Mounted at /content/drive
HDFS.log
anomaly_label.csv


In [2]:
import pandas as pd
import re
from collections import defaultdict

# Load labels
labels = pd.read_csv(LABEL_FILE)

labels.head()

,BlockId,Label
0,blk_-1608999687919862906,Normal
1,blk_7503483334202473044,Normal
2,blk_-3544583377289625738,Anomaly
3,blk_-9073992586687739851,Normal
4,blk_7854771516489510256,Normal


In [3]:
block_logs = defaultdict(list)

pattern = re.compile(r"(blk_-?\d+)")

with open(LOG_FILE, "r") as file:

    for line in file:

        match = pattern.search(line)

        if match:
            block_id = match.group(1)
            block_logs[block_id].append(line.strip())

print("Total Parsed Blocks:", len(block_logs))

Total Parsed Blocks: 575061


In [4]:
block_warn = {}

for block, logs in block_logs.items():

    has_warn = any(" WARN " in log for log in logs)

    block_warn[block] = has_warn

In [5]:
labels["Contains_WARN"] = labels["BlockId"].map(block_warn)

labels["Contains_WARN"] = labels["Contains_WARN"].fillna(False)

labels.head()

,BlockId,Label,Contains_WARN
0,blk_-1608999687919862906,Normal,False
1,blk_7503483334202473044,Normal,False
2,blk_-3544583377289625738,Anomaly,True
3,blk_-9073992586687739851,Normal,False
4,blk_7854771516489510256,Normal,True


In [6]:
pd.crosstab(
    labels["Label"],
    labels["Contains_WARN"],
    margins=True
)

Contains_WARN,False,True,All
Label,,,
Anomaly,8718,8120,16838
Normal,432103,126120,558223
All,440821,134240,575061


In [7]:
anomaly_without_warn = labels[
    (labels["Label"] == "Anomaly") &
    (labels["Contains_WARN"] == False)
]

print("Anomalous Blocks WITHOUT WARN :", len(anomaly_without_warn))

Anomalous Blocks WITHOUT WARN : 8718


In [8]:
normal_with_warn = labels[
    (labels["Label"] == "Normal") &
    (labels["Contains_WARN"] == True)
]

print("Normal Blocks WITH WARN :", len(normal_with_warn))

Normal Blocks WITH WARN : 126120


In [9]:
pd.crosstab(
    labels["Label"],
    labels["Contains_WARN"],
    margins=True
)

Contains_WARN,False,True,All
Label,,,
Anomaly,8718,8120,16838
Normal,432103,126120,558223
All,440821,134240,575061


In [10]:
print("Anomalous Blocks WITHOUT WARN :", len(anomaly_without_warn))
print("Normal Blocks WITH WARN :", len(normal_with_warn))

Anomalous Blocks WITHOUT WARN : 8718
Normal Blocks WITH WARN : 126120
